# Data Governance and Provenance

This lab introduces practical data governance and lineage tracking for analytical workflows, accompanying [`governance-lab.md`](governance-lab.md) and [`docs/DATA-LINEAGE.md`](../docs/DATA-LINEAGE.md).

## What is Data Governance?

**Data Governance** is the framework of policies, roles, and processes that ensure data is accurate, available, consistent, and secure across its lifecycle.

**Data Provenance** (lineage) answers: *where did this data come from, how was it transformed, and who consumed it?*

### The Five Pillars of Data Governance

| Pillar | What it means in practice |
|---|---|
| **Quality** | Null checks, range validation, schema enforcement, data contracts |
| **Lineage** | Track source → transformation → destination across all hops |
| **Catalog** | Discoverable metadata (column descriptions, owners, tags, retention policies) |
| **Access Control** | Role-based permissions, column-level masking, PII tokenization |
| **Auditability** | Query logs, immutable access records, and compliance verification |

## 1. Provenance Metadata Tracking

In production data architectures (e.g., Lakehouse Bronze → Silver → Gold layers), each step must record provenance metadata:
- **`run_id`**: Unique execution identifier
- **`source` & `destination`**: Storage URIs or table namespaces
- **`transform`**: Transformation job or step applied
- **`rows`**: Input and output record counts
- **`schema_version`**: Contract schema version
- **`content_hash`**: Verification checksum for tamper detection and idempotency
- **`timestamp`**: UTC execution timestamp

In [ ]:
import hashlib
import datetime
import json

def record_lineage(source, destination, row_count, transform='none', schema_version='1.0'):
    '''Capture provenance metadata for a single data movement step.'''
    timestamp = datetime.datetime.now(datetime.timezone.utc).isoformat()
    run_id = hashlib.md5(f'{source}{destination}{timestamp}'.encode()).hexdigest()[:8]
    content_hash = hashlib.sha256(f'{source}{row_count}'.encode()).hexdigest()[:16]
    return {
        'run_id':          run_id,
        'source':          source,
        'destination':     destination,
        'transform':       transform,
        'rows':            row_count,
        'schema_version':  schema_version,
        'loaded_at':       timestamp,
        'content_hash':    content_hash,
    }

# Simulate a multi-step pipeline with lineage captured at each hop
lineage_log = [
    record_lineage('classicmodels.orders',          's3://lake/raw/orders/',       row_count=326, transform='extract'),
    record_lineage('s3://lake/raw/orders/',          's3://lake/clean/orders/',     row_count=318, transform='clean+dedupe'),
    record_lineage('s3://lake/clean/orders/',        'warehouse.fact_orders',       row_count=318, transform='dimensional_load'),
]

print('Data Lineage Log:')
print(f'{"Step":<6} {"Source":<35} {"Destination":<35} {"Rows":>6}  Transform')
print('-' * 100)
for i, entry in enumerate(lineage_log, 1):
    print(f'{i:<6} {entry["source"]:<35} {entry["destination"]:<35} {entry["rows"]:>6}  {entry["transform"]}')

rows_dropped = lineage_log[0]['rows'] - lineage_log[1]['rows']
drop_pct = (rows_dropped / lineage_log[0]['rows']) * 100
print(f'\nData Observability: {rows_dropped} rows dropped during cleaning ({drop_pct:.1f}% data rejection rate)')

## 2. Lineage Observability & Audit Payloads

Lineage logs are often emitted to an observability topic, data catalog (e.g. OpenLineage, DataHub), or logged to an audit table.

In [ ]:
# Serialize lineage record for catalog ingestion or API audit logging
print(json.dumps(lineage_log[-1], indent=2))